In [3]:
from fasthtml.common import *

app, router = fast_app(live=False)

In [4]:
import os, uuid
from openai import OpenAI

# OpenCode Go requires (opencode.ai/docs/go):
#   - a coding-agent User-Agent, not the generic SDK one
#   - a stable x-opencode-session per conversation
llm_client = OpenAI(
    base_url="https://opencode.ai/zen/go/v1",
    api_key=os.environ.get("OPENCODE_GO_KEY") or "unset",
    default_headers={"User-Agent": "finityagent/1.0"},
)
chat_session = str(uuid.uuid4())

In [5]:
models = llm_client.models.list()
model_ids = [m.id for m in models.data]
# print(model_ids)
model_ids

['deepseek-v4-flash',
 'deepseek-v4-flash-vision-exp',
 'deepseek-flash',
 'deepseek-v4.1-flash',
 'deepseek-v4-pro',
 'glm-5.1',
 'glm-5.2',
 'glm-5.3',
 'grok-4.6',
 'muse-spark-1.2-contributor',
 'muse-spark-1.3-contributor',
 'glm-5.3-flash',
 'omen-alpha',
 'gpt-5.6-luna',
 'hy3',
 'hy4-preview',
 'kimi-k2.6',
 'kimi-k2.7-code',
 'kimi-k3',
 'mimo-v2.5',
 'mimo-v2.5-pro',
 'minimax-m2.5',
 'minimax-m2.7',
 'minimax-m3',
 'longcat-2.0',
 'qwen3.6-plus',
 'qwen3.7-max',
 'qwen3.8-max',
 'qwen3.8-flash',
 'qwen3.7-plus']

In [6]:
current_model = "glm-5.3-flash"
assert current_model in model_ids
if not os.environ.get("OPENCODE_GO_KEY"):
    print("Set OPENCODE_GO_KEY in .env (opencode.ai/auth) and re-run to test chat")
else:
    reply = llm_client.chat.completions.create(
        model=current_model,
        messages=[{"role": "user", "content": "Say hello in one sentence."}],
        extra_headers={"x-opencode-session": chat_session},
    )
    print(reply.choices[0].message.content)

Hello, it's wonderful to meet you!


# Agent loop — one tool: bash
Chat history → model → tool call → `subprocess` → result → model. The agent answers in plain text or an HTML artifact.
disk: 15 cells


In [9]:
# Tool definition
BASH_TOOL = {
    "type": "function",
    "function": {
        "name": "bash",
        "description": "Run a bash command and return stdout+stderr. Use for file ops, code execution, data processing, websearch, cronjobs — everything.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "The bash command to run"},
            },
            "required": ["command"],
        },
    },
}

import subprocess

def run_bash(command: str, timeout: int = 60) -> str:
    """Execute one bash command, return combined stdout/stderr."""
    try:
        r = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout)
        out = (r.stdout + ("\n[stderr]\n" + r.stderr if r.stderr else "")).strip()
        return out or f"(exit {r.returncode}, no output)"
    except subprocess.TimeoutExpired:
        return f"(timed out after {timeout}s)"

# smoke test
print(run_bash("echo hi && pwd"))

hi
/Users/fakhirali/Code/FinityAgent


In [79]:
# Agent loop
import json as _json

SYSTEM_PROMPT = """You are Finity Agent, an agent with exactly one tool: bash. You will use it
for everything such as file ops, code execution, websearch, external integrations, scheduling tasks etc. Respond in html formatting instead of markdown please."""

def chat(messages: list, max_turns: int = 8, model: str | None = None) -> list:
    """Run the agent loop; returns the full message history (mutated in place)."""
    for _ in range(max_turns):
        reply = llm_client.chat.completions.create(
            model=model or current_model,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}] + messages,
            tools=[BASH_TOOL],
            extra_headers={"x-opencode-session": chat_session},
        )
        msg = reply.choices[0].message
        if not msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content})
            return messages
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [{"id": tc.id, "type": "function",
                                         "function": {"name": tc.function.name,
                                                      "arguments": tc.function.arguments}}
                                        for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            if tc.function.name != "bash":
                result = f"(unknown tool {tc.function.name})"
            else:
                args = _json.loads(tc.function.arguments)
                # print(f"$ {args['command']}")
                result = run_bash(args["command"])
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return messages

# Try it
history = [{"role": "user", "content": "Use bash to tell me how many files are in the current directory and what they are."}]
chat(history)
print()
print("--- final answer ---")
print(history[-1]["content"][:1500])


--- final answer ---
Here's the overview of the current directory — it has <strong>6 non-hidden files</strong> (plus 15 hidden files/directories):

<h3>Visible files (6)</h3>
<ul>
<li><code>AGENTS.md</code> — documentation file (1.2 KB)</li>
<li><code>FinityAgent.ipynb</code> — Jupyter notebook (18.7 KB)</li>
<li><code>finityagent.py</code> — Python script (2.3 KB)</li>
<li><code>pyproject.toml</code> — project config (56 B)</li>
<li><code>README.md</code> — readme (1.3 KB)</li>
</ul>

Wait, that's 5 👇

<h3>Hidden entries (10 files/dirs)</h3>
<ul>
<li><code>.env</code>, <code>.gitignore</code>, <code>.jupyter_ystore.db</code>, <code>.sesskey</code> (files)</li>
<li><code>__pycache__/</code>, <code>.git/</code>, <code>.ipynb_checkpoints/</code>, <code>.jupyter/</code>, <code>.playwright-cli/</code>, <code>.pytest_cache/</code>, <code>.venv/</code> (directories)</li>
</ul>

So: <strong>6 regular files</strong> in `ls | wc -l` terms... actually `ls | wc -l` returned 6 — that's the 5 file

In [18]:
while True:
    user = input("\nyou> ").strip()
    if not user:
        continue
    if user in {"q", "quit", "exit"}:
        print("bye")
        break
    history.append({"role": "user", "content": user})
    chat(history)
    print(f"\nagent> {history[-1]['content']}")


you>  Look at the .agents/skills directory please and checkout the websearch skill


$ ls -la .agents/skills/ 2>&1; echo ---; ls -la .agents/skills/websearch/ 2>&1
$ find . -type d -name "skills" -not -path "./.venv/*" 2>/dev/null; find / -maxdepth 6 -type d -name "websearch" 2>/dev/null | head
$ find / -maxdepth 7 -name "*websearch*" 2>/dev/null | grep -v ".venv" | head; ls ~/.agents 2>&1; ls ~/.finity 2>&1
$ ls -la ~/.claude/skills/freeunlimited-websearch/; echo ---; for f in ~/.claude/skills/freeunlimited-websearch/*; do echo "=== $f ==="; cat "$f"; done

agent> Note: there's no `.agents/skills` directory in this project — but I found the websearch skill at `~/.claude/skills/freeunlimited-websearch/`. Here's what it contains:

**Files:**
- `SKILL.md` — skill metadata & docs
- `search.py` — the search implementation
- `README.md` — quick install guide
- `LICENSE` — MIT license

**What the skill does:**
- Free, unlimited web search using **DuckDuckGo** via the `ddgs` Python package — no API key or rate limits
- `search.py` takes a query as an argument, runs `DDGS().te


you>  Yeah search for what people are saying about eric ries' new book


$ pip list 2>/dev/null | grep -i ddgs || pip install -q ddgs 2>&1 | tail -1; python3 ~/.claude/skills/freeunlimited-websearch/search.py "Eric Ries new book 2025"
$ .venv/bin/pip install -q ddgs 2>&1 | tail -1; .venv/bin/python ~/.claude/skills/freeunlimited-websearch/search.py "Eric Ries new book 2025"
$ ls .venv/bin | head -20; which python3 pip3
$ pip3 install -q ddgs 2>&1 | tail -1; python3 ~/.claude/skills/freeunlimited-websearch/search.py "Eric Ries new book 2025"
$ python3 -m venv /tmp/ddgsenv && /tmp/ddgsenv/bin/pip install -q ddgs 2>&1 | tail -1; /tmp/ddgsenv/bin/python ~/.claude/skills/freeunlimited-websearch/search.py "Eric Ries new book 2025"
$ /tmp/ddgsenv/bin/python ~/.claude/skills/freeunlimited-websearch/search.py "Eric Ries Incorruptible book review reactions what people are saying" 2>/dev/null | python3 -c "import json,sys; [print(f\"- {r['title']}: {r['href']}\") for r in json.load(sys.stdin)]"
$ curl -sL "https://dougalshaw.substack.com/p/book-review-incorruptible-by


you>  q


bye


In [94]:
page = Div(
    Div(
        Div("Hi! I'm FinityAgent — one tool: bash.",
            style="align-self:flex-start;"),
        Div("What files are in this directory?",
            style="align-self:flex-end;"),
        Div("$ ls → FinityAgent.ipynb, finityagent.py, pyproject.toml, README.md",
            style="align-self:flex-start;"),
        id="chat-log",
        style="display:flex; flex-direction:column; gap:6px; flex:1;"
              " overflow-y:auto; padding:14px; line-height:1.3;",
    ),
    Form(
        Input(name="msg", placeholder="Type a message…", required=True,
              autocomplete="off",
              style="flex:1; height:50px; box-sizing:border-box;"),
        Button("Send", style="height:50px; box-sizing:border-box;"),
        style="display:flex; gap:8px; margin-top:10px;",
        hx_post="/chat", hx_target="#chat-log", hx_swap="beforeend",
        hx_on__after_request="this.reset()",
    ),
    style="display:flex; flex-direction:column; height:100vh;"
          " padding:16px; box-sizing:border-box;",
)

In [96]:
import json
history = [] 
@router("/")
def get():
    return page

# Step 1: echo the user's message back immediately...
@router("/chat")
def post(msg: str):
    
    return Div(
        msg, style="align-self:flex-end; white-space:pre-wrap;",
        hx_post="/agent", hx_trigger="load",
        hx_target="#chat-log", hx_swap="beforeend",
        hx_vals=json.dumps({"msg": msg}),
    )

# Step 2: ...the echoed div fires this on load and swaps in the agent's response
@router("/agent")
def agent(msg: str):
    global history
    if "history" not in globals():
        history = []
    history.append({"role": "user", "content": msg})
    chat(history)
    return Div(NotStr(history[-1]["content"]), style="align-self:flex-start;")

In [14]:
server = nb_serve(app, port=5437)